# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset metadata summary
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll use the Croissant API to enumerate the record sets, their fields, and columns. All references will use each entity's `@id` as per the Croissant specification.

In [ ]:
# List out all record sets in the dataset with their @id and fields

record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets are defined in the Croissant metadata.")
else:
    for rs in record_sets:
        print(f"\nRecordSet Name: {rs.name}")
        print(f"  @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}) [Type: {field.data_type}]")
        print("  Columns:")
        for col in getattr(rs, 'columns', []):
            print(f"    - {col.name} (@id: {col.id})")
        print('-'*40)

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from available record sets into pandas DataFrames
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

if not record_set_ids:
    print("No record sets to extract data from.")
else:
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet @id: {record_set_id} with {len(df)} records.")
        except Exception as e:
            print(f"Could not load records for RecordSet @id: {record_set_id}: {e}")

# Show the columns of the first record set, if present
if record_set_ids:
    first_rs_id = record_set_ids[0]
    df = dataframes[first_rs_id]
    print(f"\nFirst 5 rows for RecordSet @id: {first_rs_id}")
    print("Columns:", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalizing, and grouping using fields by their `@id`. We'll select a numeric field, filter, normalize, and group as appropriate.

You may need to adapt field and group choices based on what's present in the loaded DataFrame. If no record sets are present, this cell will describe anticipated usage.

In [ ]:
# Example EDA - Select a numeric field in the first record set
import numpy as np

if not record_set_ids or dataframes[first_rs_id].empty:
    print("No data available for EDA. Please check the dataset record sets.")
else:
    df = dataframes[first_rs_id]
    # Find a numeric field (by dtype or by field definition)
    numeric_column = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_column = col
            break
    if numeric_column is None:
        print("No numeric fields were found in the DataFrame.")
    else:
        print(f\"Using numeric field '@id': {numeric_column}\")

        # Filter records (example: values greater than the median)
        threshold = df[numeric_column].median()
        filtered_df = df[df[numeric_column] > threshold].copy()
        print(f"Filtered records with {numeric_column} > {threshold}:")
        display(filtered_df.head())
        
        filtered_df[f"{numeric_column}_normalized"] = (filtered_df[numeric_column] - filtered_df[numeric_column].mean()) / filtered_df[numeric_column].std()
        print(f"Normalized {numeric_column} for filtered records:")
        display(filtered_df[[numeric_column, f"{numeric_column}_normalized"]].head())

        # Choose a group field - pick a categorical column if available
        group_field = None
        for col in df.columns:
            if df[col].dtype == 'object' and col != numeric_column:
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by field '@id': {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_column].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")

## 5. Visualization
Visualize the distribution of a numeric field and/or show relationships with a categorical field, if present in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids or dataframes[first_rs_id].empty or numeric_column is None:
    print("Insufficient data for visualization.")
else:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_column], kde=True)
    plt.title(f"Distribution of {numeric_column}")
    plt.xlabel(numeric_column)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_column, data=df)
        plt.title(f"{numeric_column} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library for structured, reproducible data access and processing using a Croissant schema. We loaded the dataset, explored metadata, programmatically referenced record sets and fields by `@id`, and performed basic EDA and visualizations. 

You can adapt this notebook to perform more advanced processing, modeling, or visualization on the FAIR² dataset or any other dataset described by a Croissant schema.